In [0]:
%sql
describe workspace.pj_sales.tb_sales_silver

In [0]:
%sql
create table if not exists 
  workspace.pj_sales.tb_dim_store_gold (
     _sk_store bigint generated by default as identity
    ,store_id int
    ,store_name string
    ,store_group string
    ,valid_from timestamp
    ,valid_to timestamp
  )

In [0]:
%sql
insert into workspace.pj_sales.tb_dim_store_gold
values (-1, null, 'unknown', 'unknown', null, null)

In [0]:
%sql
create or replace temp view vw_dim_store as(
  with dedup as (
    select
       store_id
      ,store_name
      ,store_group
      ,row_number() over (
        partition by 
           store_id
        order by 
           sale_date desc
          ,ingestion_timestamp desc
      )                                                             as order_by_date
    from workspace.pj_sales.tb_sales_silver
    where part_quantity > 0
  )
  select
     store_id
    ,store_name
    ,store_group
    ,from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo')   as valid_from
    ,null                                                           as valid_to
  from dedup
  where order_by_date = 1
)

In [0]:
%sql
create or replace temp view vw_dim_store_merge as
select
  vst.*,
  'u' as action
from vw_dim_store vst -- Aqui o registro serve para ser atualizado
union all
select
  vst.*,
  'i' as action
from vw_dim_store vst -- Aqui o registro serve para ser inserido

In [0]:
%sql
merge into workspace.pj_sales.tb_dim_store_gold tst
using vw_dim_store_merge vstm
on  tst.store_id = vstm.store_id
and tst.valid_to is null
and vstm.action = 'u'
when matched and vstm.action = 'u' then
  update set
    tst.valid_to = vstm.valid_from
when not matched and vstm.action = 'i' then
  insert (
     store_id
    ,store_name
    ,store_group
    ,valid_from
    ,valid_to
  )
  values (
     vstm.store_id
    ,vstm.store_name
    ,vstm.store_group
    ,vstm.valid_from
    ,vstm.valid_to
  )


In [0]:
%sql
select * from pj_sales.tb_dim_store_gold